# Lab 4 — Foundry Hosted Agents

Deploy an **Agent Framework** agent as a containerized, Microsoft-managed **hosted agent** on **Microsoft Foundry Agent Service**. The platform handles scaling, session-state persistence, security, and lifecycle management, so you focus on the agent's logic.

With the hosting integration, you can take any `Agent` (or workflow) and expose it through the Foundry **Responses** or **Invocations** protocol with minimal code.

> This lab uses **[`uv`](https://docs.astral.sh/uv/)** to create and manage the virtual environment on **Python 3.13**.

**Source:** [Foundry Hosted Agents (Agent Framework docs)](https://learn.microsoft.com/en-us/agent-framework/hosting/foundry-hosted-agent?pivots=programming-language-python) · [Quickstart: Deploy your first hosted agent](https://learn.microsoft.com/en-us/azure/foundry/agents/quickstarts/quickstart-hosted-agent?pivots=azd)

---

### What you'll do

1. Set up an isolated project + venv with `uv` (Python 3.13)
2. Write an agent exposed via the **Responses** protocol (and see the **Invocations** alternative)
3. Run and test the agent **locally**
4. **Deploy** the agent to Foundry Agent Service with `azd`
5. **Invoke** the deployed agent, then **clean up**

> ⏱️ ~25 min · Requires an Azure subscription with Foundry access.


## When to use hosted agents

Choose Foundry hosted agents when you want:

- **Managed infrastructure** — no containers, web servers, or scaling rules to configure yourself.
- **Built-in session management** — the platform persists `$HOME` and uploaded files across turns and idle periods.
- **Dedicated agent identity** — every deployed agent gets its own Entra identity for secure access to models, tools, and downstream services.
- **OpenAI-compatible endpoints** — clients can interact with your agent using any OpenAI-compatible SDK through the Responses protocol.

> ⚠️ Foundry hosted agents are currently in **preview**. See the [hosted agents docs](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/hosted-agents#limits-pricing-and-availability-preview) for the latest availability, limits, and pricing.

## Prerequisites

- An **Azure subscription** with a [Microsoft Foundry](https://learn.microsoft.com/en-us/azure/foundry/) project and a model deployment (for example `gpt-4.1-mini`).
- **Python 3.13 or later** — [download](https://www.python.org/downloads/).
- **[`uv`](https://docs.astral.sh/uv/)** for environment + dependency management.
- **[Azure CLI](https://learn.microsoft.com/en-us/cli/azure/install-azure-cli)** installed and authenticated (`az login`) for local testing.
- **[Azure Developer CLI (`azd`)](https://learn.microsoft.com/en-us/azure/developer/azure-developer-cli/install-azd) 1.25.3+** with the Foundry agent extension (used for provisioning, running, and deploying).

Roles: `Foundry Project Manager` at project scope (existing project) or `Owner` at resource-group scope (to create a new project).


## Step 1 — Set up the project and venv with `uv` (Python 3.13)

We'll scaffold a self-contained `hosted-agent/` project folder, pin it to **Python 3.13**, and let `uv` create the virtual environment.

If you don't have `uv` yet, install it first:

```bash
# macOS / Linux
curl -LsSf https://astral.sh/uv/install.sh | sh

# Windows (PowerShell)
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
```

> The cells below use `!` to run shell commands from the notebook. `uv` reads the pinned interpreter from `.python-version` and manages an isolated `.venv` inside `hosted-agent/`.


In [8]:
# Verify uv is installed and initialize the hosted-agent project pinned to Python 3.13
!uv --version

# Create an isolated project folder and pin the interpreter to 3.13
!uv init --python 3.13 --no-workspace  hosted-agent 2>/dev/null  || echo "hosted-agent already initialized"

# Create the virtual environment with Python 3.13 (uv downloads it if needed)
!cd hosted-agent && uv venv --python 3.13 --clear

# Show the pinned version
!cat hosted-agent/.python-version

uv 0.8.11 (f892276ac 2025-08-14)
hosted-agent already initialized
Using CPython 3.13.14 interpreter at: /opt/homebrew/opt/python@3.13/bin/python3.13
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate
3.13


In [9]:
# Add the hosting dependencies to the project (records them in pyproject.toml + uv.lock)
!cd hosted-agent && uv add agent-framework agent-framework-foundry-hosting azure-identity

# Confirm they resolved into the 3.13 venv
!cd hosted-agent && uv pip list | grep -i "agent-framework\|azure-identity"

Resolved 202 packages in 5ms
Installed 188 packages in 1.19s2                            
 + a2a-sdk==1.1.0
 + ag-ui-protocol==0.1.19
 + agent-framework==1.10.0
 + agent-framework-a2a==1.0.0b260604
 + agent-framework-ag-ui==1.0.0rc7
 + agent-framework-anthropic==1.0.0b260630
 + agent-framework-azure-ai-search==1.0.0b260630
 + agent-framework-azure-cosmos==1.0.0b260521
 + agent-framework-azurefunctions==1.0.0b260630
 + agent-framework-bedrock==1.0.0b260630
 + agent-framework-chatkit==1.0.0b260528
 + agent-framework-claude==1.0.0b260609
 + agent-framework-copilotstudio==1.0.0b260521
 + agent-framework-core==1.10.0
 + agent-framework-declarative==1.0.0rc2
 + agent-framework-devui==1.0.0b260630
 + agent-framework-durabletask==1.0.0b260630
 + agent-framework-foundry==1.10.0
 + agent-framework-foundry-hosting==1.0.0a260630
 + agent-framework-foundry-local==1.0.0b260521
 + agent-framework-github-copilot==1.0.0rc2
 + agent-framework-lab==1.0.0b251024
 + agent-framework-mem0==1.0.0b260609
 + ag

## Step 2 — Configure environment variables

The agent reads its Foundry project endpoint and model deployment name from the environment. Locally you set these yourself; when deployed, Foundry **injects them automatically** into the container.

| Variable | Description |
| --- | --- |
| `FOUNDRY_PROJECT_ENDPOINT` | Endpoint URL for the Foundry project, e.g. `https://<account>.services.ai.azure.com/api/projects/<project>` |
| `AZURE_AI_MODEL_DEPLOYMENT_NAME` | The model deployment name, e.g. `gpt-4.1-mini` |
| `APPLICATIONINSIGHTS_CONNECTION_STRING` | (Injected at runtime) telemetry connection string |

Fill in your values below — this writes a `.env` file into the project folder.


In [10]:
from pathlib import Path

# from pathlib import Path

# # 👇 Replace with your own Foundry project values
# FOUNDRY_PROJECT_ENDPOINT = "https://<account>.services.ai.azure.com/api/projects/<project>"
# AZURE_AI_MODEL_DEPLOYMENT_NAME = "gpt-4.1-mini"

# env_path = Path("hosted-agent/.env")
# env_path.write_text(
#     f"FOUNDRY_PROJECT_ENDPOINT={FOUNDRY_PROJECT_ENDPOINT}\n"
#     f"AZURE_AI_MODEL_DEPLOYMENT_NAME={AZURE_AI_MODEL_DEPLOYMENT_NAME}\n"
# )
# print(f"Wrote {env_path.resolve()}")
# print(env_path.read_text())

env_path = Path("hosted-agent/.env")

if not env_path.exists():
    raise FileNotFoundError(f"Missing .env file at: {env_path.resolve()}")

env_vars = {}
for line in env_path.read_text().splitlines():
    line = line.strip()
    if not line or line.startswith("#") or "=" not in line:
        continue
    key, value = line.split("=", 1)
    env_vars[key.strip()] = value.strip()

FOUNDRY_PROJECT_ENDPOINT = env_vars.get("FOUNDRY_PROJECT_ENDPOINT")
AZURE_AI_MODEL_DEPLOYMENT_NAME = env_vars.get("AZURE_AI_MODEL_DEPLOYMENT_NAME")

print(f"Loaded {env_path.resolve()}")
print("FOUNDRY_PROJECT_ENDPOINT =", FOUNDRY_PROJECT_ENDPOINT)
print("AZURE_AI_MODEL_DEPLOYMENT_NAME =", AZURE_AI_MODEL_DEPLOYMENT_NAME)

Loaded /Users/jinle/Repos/_Demos/AgentFrameworkLabs/hosted-agent/.env
FOUNDRY_PROJECT_ENDPOINT = https://aifoundry825233136833-resource.services.ai.azure.com/api/projects/aifoundry825233136833
AZURE_AI_MODEL_DEPLOYMENT_NAME = gpt-5.3-codex


## Step 3 — Write the agent (Responses protocol)

The **Responses** protocol is the recommended starting point for most agents. It exposes an OpenAI-compatible `/responses` endpoint, and the platform manages conversation history, streaming, and session lifecycle automatically.

Key points:

- `FoundryChatClient` connects the agent to your Foundry project + model deployment using `DefaultAzureCredential`.
- `ResponsesHostServer` wraps the agent and exposes it through the Foundry Responses protocol.
- Setting `store` to `False` in `default_options` avoids duplicating conversation history, since the hosting infrastructure manages history for you.

The next cell writes `hosted-agent/main.py`.


In [11]:
%%writefile hosted-agent/main.py
import os

from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient
from agent_framework_foundry_hosting import ResponsesHostServer
from azure.identity import DefaultAzureCredential

client = FoundryChatClient(
    project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=DefaultAzureCredential(),
)

agent = Agent(
    client=client,
    instructions="You are a helpful AI assistant.",
    default_options={"store": False},
)

server = ResponsesHostServer(agent)

if __name__ == "__main__":
    server.run()

Overwriting hosted-agent/main.py


### (Alternative) Invocations protocol

The **Invocations** protocol gives you full control over the HTTP request and response. Use it when you need custom payloads, non-conversational processing, or streaming protocols that aren't OpenAI-compatible.

For a lightweight setup, swap `ResponsesHostServer` for `InvocationsHostServer` — it wraps the agent the same way and handles session management automatically:

```python
import os

from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient
from agent_framework_foundry_hosting import InvocationsHostServer
from azure.identity import DefaultAzureCredential

client = FoundryChatClient(
    project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=DefaultAzureCredential(),
)

agent = Agent(
    client=client,
    instructions="You are a friendly assistant. Keep your answers brief.",
    default_options={"store": False},
)

server = InvocationsHostServer(agent)
server.run()
```

For full control over request handling, use `InvocationAgentServerHost` from `azure.ai.agentserver.invocations` directly and implement your own `@app.invoke_handler`.

> ⚠️ In-memory session stores in custom handlers are lost on restart. Use durable storage (for example, Cosmos DB — see Lab 1a) in production.

**This lab continues with the Responses protocol (`main.py`) above.**


## Step 4 — Run the agent locally

Authenticate with Azure so `DefaultAzureCredential` can reach your Foundry project, then start the server.

```bash
az login
```

You have two options to run locally:

**Option A — run `main.py` directly with `uv`** (uses the venv + `.env` you set up):

```bash
cd hosted-agent
uv run --env-file .env python main.py
```

The agent listens on `http://localhost:8088/` and serves the Responses protocol at `POST /responses`.

**Option B — use the `azd` agent tooling** (recommended: it also sets you up for deployment in Step 5).

First initialize the project as a **hosted agent** with `azd ai agent init`. This is the command that generates the `azure.ai.agent` service in `azure.yaml`, and it interactively walks you through selecting/creating the **Foundry project**, **model deployment**, and **container registry**:

```bash
cd hosted-agent
azd ai agent init
```

The interactive flow prompts for: **Agent name**, **Foundry Project** (create new or use existing), **Tenant**, **Subscription**, **Location**, **Model** (e.g. `gpt-4.1-mini`), **Model Version/SKU/Capacity**, and **Deployment name**. Keep the generated `main.py`, or replace it with the Responses agent from Step 3.

> 💡 You can also point `azd ai agent init` at a sample manifest with `-m <url-or-path> --deploy-mode code` to scaffold from a known-good template — but running it with no flags starts the same interactive wizard.

Then run the agent host locally (creates a venv, installs deps, launches the agent, and opens the agent inspector in your browser):

```bash
azd ai agent run
```

Send a test prompt from another terminal (see the next cell).

> ⚠️ **Don't use plain `azd init`.** It scans the folder and scaffolds a generic **Azure Container Apps** `azure.yaml` (`host: containerapp`) with **no `azure.ai.agent` service** — so `azd ai agent run` fails with `ERROR: no azure.ai.agent service found in azure.yaml`. If you already ran `azd init`, delete the generated files first and re-init as an agent:
>
> ```bash
> cd hosted-agent
> rm -f azure.yaml next-steps.md
> azd ai agent init
> ```

---

### 🩺 Two log lines you can safely ignore when running locally

When you start the server off-Azure, you'll see these in the logs. **Neither stops the agent:**

1. **`Inbound GET / completed with status 404`** — a browser/health probe hit the root path `/`. The agent only serves `POST /responses`, so `/` correctly returns 404. Not an error.

2. **`ConnectTimeout ... 169.254.169.254 .../metadata/instance/compute`** — the OpenTelemetry **`azure_vm` resource detector** probing the Azure Instance Metadata Service (IMDS) to tag telemetry with VM attributes. That IP only exists **on an Azure VM**, so on your laptop it times out after 0.2s and is ignored. It disappears once deployed to Foundry.

**To silence the IMDS probe locally**, exclude the `azure_vm` detector via `OTEL_EXPERIMENTAL_RESOURCE_DETECTORS`. Either add it to your `.env`:

```dotenv
OTEL_EXPERIMENTAL_RESOURCE_DETECTORS=otel,host,os,process,azure_app_service,azure_functions
```

…or set it inline when you run:

```bash
cd hosted-agent
OTEL_EXPERIMENTAL_RESOURCE_DETECTORS=otel,host,os,process uv run --env-file .env python main.py
```


In [12]:
# With the server running (Option A or B) in another terminal, send a test prompt to the /responses endpoint.
!curl -sS -H "Content-Type: application/json" \
    -X POST http://localhost:8088/responses \
    -d '{"input": "Write a haiku about deploying cloud applications.", "stream": false}'

# Or, if you used `azd ai agent run`:
#   azd ai agent invoke --local "Hello!"

curl: (7) Failed to connect to localhost port 8088 after 0 ms: Couldn't connect to server


## Step 4b — Add the deployment files

Because `azure.yaml` uses `language: docker` with `remoteBuild: true`, azd builds your agent **from a `Dockerfile`**. A hosted-agent project needs four small files alongside `main.py`:

| File | Purpose |
| --- | --- |
| `Dockerfile` | How to build the container image (base image + `pip install` + `CMD`). **Without it, deploy fails with `failed retrieving package result details`.** |
| `requirements.txt` | Dependencies installed into the image. Use the lean **`agent-framework-foundry`** package — *not* the full `agent-framework` meta-package (~200 deps). |
| `.dockerignore` | Keeps `.venv/`, `.azure/`, caches out of the build context (avoids `archive/tar: write too long`). |
| `.azdignore` | Keeps agent-config files out of the azd deployment package. |

The next cells write these into `hosted-agent/`.


In [13]:
%%writefile hosted-agent/Dockerfile
FROM python:3.13-slim

WORKDIR /app

COPY . user_agent/
WORKDIR /app/user_agent

RUN if [ -f requirements.txt ]; then \
        pip install --no-cache-dir -r requirements.txt; \
    else \
        echo "No requirements.txt found"; \
    fi

EXPOSE 8088

CMD ["python", "main.py"]

Overwriting hosted-agent/Dockerfile


In [14]:
%%writefile hosted-agent/requirements.txt
# Minimal deps for a Foundry hosted agent using the Responses protocol.
# `agent-framework-foundry` provides `agent_framework.Agent` + `agent_framework.foundry.FoundryChatClient`.
# Avoid the full `agent-framework` meta-package (pulls bedrock/anthropic/google/... ~200 deps).
agent-framework-foundry
agent-framework-foundry-hosting>=1.0.0a260630
azure-identity

Overwriting hosted-agent/requirements.txt


In [15]:
%%writefile hosted-agent/.dockerignore
# Keep the build context small; excludes huge/mutating dirs and secrets.
.venv/
venv/
env/
.azure/
.env
*.env
__pycache__/
*.py[cod]
*.egg-info/
.mypy_cache/
.pytest_cache/
.ruff_cache/
.ipynb_checkpoints/
.git/
.gitignore
.vscode/
.DS_Store
next-steps.md

Overwriting hosted-agent/.dockerignore


In [16]:
%%writefile hosted-agent/.azdignore
agent.manifest.yaml
agent.yaml
.env.example

Overwriting hosted-agent/.azdignore


## Step 5 — Deploy to Foundry Agent Service

By now `azd ai agent init` (Step 4) has generated the `azure.ai.agent` service in `azure.yaml`, and Step 4b added the `Dockerfile`, `requirements.txt`, `.dockerignore`, and `.azdignore` that the container build needs. Provision + deploy:

```bash
cd hosted-agent

# 1) Provision Foundry project, model deployment, App Insights, and container registry (if needed)
azd provision

# 2) Package as a container image, push to ACR, and deploy to Foundry Agent Service
azd deploy
```

> If you haven't initialized the agent project yet, run `azd ai agent init` first (see Step 4). Plain `azd init` will **not** work — it produces a Container Apps config with no `azure.ai.agent` service. If your `azure.yaml` still contains a leftover `resources:` block with `type: host.containerapp` (from a prior `azd init`), remove it or regenerate the file with `azd ai agent init`.

When `azd deploy` finishes, it prints links to the **agent playground** (portal) and the **agent endpoint**:

```output
- Agent playground (portal): https://ai.azure.com/.../build/agents/<agent-name>/build?version=1
- Agent endpoint: https://ai-account-<name>.services.ai.azure.com/api/projects/<project>/agents/<agent-name>/versions/1
```

> Foundry injects `FOUNDRY_PROJECT_ENDPOINT`, `AZURE_AI_MODEL_DEPLOYMENT_NAME`, and `APPLICATIONINSIGHTS_CONNECTION_STRING` into the container automatically at runtime.

### Step 5b — If the agent version fails with an image-pull error

If `azd deploy` builds + pushes the image but the agent version fails (and invoking returns):

```text
agent_version_failed: [ImageError] Container registry authentication failed.
Verify the workspace managed identity has AcrPull permissions on the target registry.
```

…then Foundry's **managed identity** can't pull the image from ACR. Common when you deploy into an **existing** Foundry project while azd creates a **new** registry — azd wires the pull role only for projects it creates.

**Fix** (next cell): grant `AcrPull` to the Foundry **account *and* project** system-assigned identities, and enable Entra/ARM-token auth on the registry.

> ⚠️ Two gotchas that make this *look* unfixed:
> - **Propagation delay** — ACR role assignments can take **5–10 minutes** to take effect. Redeploying immediately still fails.
> - **Failed versions don't self-heal** — each `azd deploy` creates a *new* version. After the role propagates, run `azd deploy` again to create a fresh version that pulls successfully, then `azd ai agent invoke`.


In [ ]:
# Grant Foundry's managed identity permission to pull the agent image from ACR.
# Grants AcrPull to BOTH the account and project system-assigned identities (belt-and-suspenders),
# enables Entra/ARM token auth on the registry, and verifies the result.
# Uses your local `az` login — run in a terminal if the notebook kernel is sandboxed.
import subprocess, shlex
from pathlib import Path

# --- Load azd environment values ---
azd_env = next(Path("hosted-agent/.azure").glob("*/.env"))
cfg = {}
for line in azd_env.read_text().splitlines():
    if "=" in line and not line.startswith("#"):
        k, v = line.split("=", 1)
        cfg[k.strip()] = v.strip().strip('"')

ACR_NAME = cfg["AZURE_CONTAINER_REGISTRY_ENDPOINT"].split(".")[0]
PROJECT_ID = cfg["AZURE_AI_PROJECT_ID"]
ACCOUNT_NAME = cfg["AZURE_AI_ACCOUNT_NAME"]
ACCOUNT_RG = cfg["AZURE_RESOURCE_GROUP"]
print(f"Registry: {ACR_NAME}\nAccount:  {ACCOUNT_NAME} (rg={ACCOUNT_RG})\nProject:  {PROJECT_ID}")

def sh(cmd, quiet=False):
    out = subprocess.run(shlex.split(cmd), capture_output=True, text=True)
    res = (out.stdout or out.stderr).strip()
    if not quiet:
        print(f"$ {cmd}\n{res}\n")
    return res

acr_id = sh(f"az acr show -n {ACR_NAME} --query id -o tsv", quiet=True)

# --- Resolve BOTH candidate pull identities ---
acct_mi = sh(f"az cognitiveservices account show -n {ACCOUNT_NAME} -g {ACCOUNT_RG} --query identity.principalId -o tsv", quiet=True)
proj_mi = sh(f'az resource show --ids "{PROJECT_ID}" --query identity.principalId -o tsv', quiet=True)
print(f"Account MI: {acct_mi or '(none)'}\nProject MI: {proj_mi or '(none)'}\n")

# --- Grant AcrPull to each identity that exists (idempotent) ---
for mi in {m for m in (acct_mi, proj_mi) if m and m != "null"}:
    sh(f'az role assignment create --assignee-object-id "{mi}" --assignee-principal-type ServicePrincipal --role AcrPull --scope "{acr_id}"')

# --- Ensure the registry accepts Entra/ARM tokens (required for MI pulls) ---
sh(f"az acr config authentication-as-arm update -n {ACR_NAME} --status enabled")

# --- Verify ---
print("=== AcrPull principals now on the registry ===")
sh(f'az role assignment list --scope "{acr_id}" --role AcrPull --query "[].principalId" -o tsv')

print("⏳ RBAC on ACR can take 5–10 min to propagate. Then run:  azd deploy  &&  azd ai agent invoke \"...\"")

## Step 6 — Invoke the deployed agent

Send a prompt to the deployed agent from the CLI:

```bash
cd hosted-agent
azd ai agent invoke "Write a haiku about deploying cloud applications."
```

You should see a haiku response within a few seconds. Optionally stream container logs while you interact:

```bash
azd ai agent monitor --follow
```

You can also test it from the **Agent playground** in the Foundry portal using the link printed by `azd deploy`.


## Step 7 — Clean up resources

Delete the resources when you're finished so you stop incurring charges.

```bash
cd hosted-agent
azd down
```

> ⚠️ `azd down` **permanently deletes every resource in the resource group** — the Foundry project, model deployments, Container Registry, Application Insights, and the hosted agent. If you provisioned into a resource group that contains other resources, those are deleted too. `azd` lists what it will delete and prompts for confirmation.

---

## What you learned

- Scaffolded an isolated hosted-agent project with **`uv`** on **Python 3.13**.
- Exposed an Agent Framework `Agent` through the Foundry **Responses** protocol (and saw the **Invocations** alternative).
- Ran and tested the agent **locally** (`uv run` / `azd ai agent run`).
- Added the container build files (`Dockerfile`, `requirements.txt`, `.dockerignore`, `.azdignore`).
- **Deployed** to Foundry Agent Service (`azd provision` → `azd deploy`) and **invoked** it.
- Cleaned up with `azd down`.

### Troubleshooting

| Issue | Solution |
| --- | --- |
| `SubscriptionNotRegistered` | `az provider register --namespace Microsoft.CognitiveServices` |
| `AuthorizationFailed` during provisioning | Request **Contributor** on the subscription or resource group |
| `AuthenticationError` / `DefaultAzureCredential` failure | `azd auth logout` then `azd auth login` (or `az login`) |
| `Connection refused` on local run | Ensure nothing else is using port **8088** |
| `azd ai agent init` fails | `azd version` (need 1.25.3+); update the extension: `azd ext upgrade microsoft.foundry` |
| `remote build failed: archive/tar: write too long` | The `.venv/` (or another large/mutating dir) is in the build context. Add a `.dockerignore` excluding `.venv/`, `.azure/`, `__pycache__/`, `.env`. |
| `failed retrieving package result details` on `azd deploy` | The container build produced no image — usually a **missing `Dockerfile`** (required for `language: docker` + `remoteBuild`). Add the `Dockerfile` + `requirements.txt` from Step 4b. |
| `Local fallback unavailable: the docker service is not running` | Only appears *after* the remote build fails. Fix the remote build (above) — you don't need Docker Desktop. Or start Docker Desktop to build locally. |
| Build installs no dependencies / `ModuleNotFoundError` at runtime | Ensure `requirements.txt` exists and lists `agent-framework-foundry` + `agent-framework-foundry-hosting` (since `.venv/` is excluded from the image). |
| `agent_version_failed` / `[ImageError] Container registry authentication failed ... AcrPull` | The project managed identity can't pull the image. Grant it **AcrPull** on the registry and enable ARM-token auth — see **Step 5b**. Common when deploying into a **pre-existing** Foundry project. |
| `image_pull_failed` (400) | Confirm the project MI has **AcrPull**/**Container Registry Repository Reader** on the ACR *and* the registry's `authentication-as-arm` policy is `enabled`. |
| `az ... could not be found in subscription` / empty principal IDs | Your active `az` subscription is wrong. Run `az account set --subscription <id>` (the one in `.azure/<env>/.env` → `AZURE_SUBSCRIPTION_ID`) before the RBAC commands. |
| `RuntimeError: hosted environment is running on protocol 1.0.0, but the agent requires protocol 2.0.0` | Bump the `responses` protocol `version` in `agent.yaml` from `1.0.0` to **`2.0.0`** (matches `agent-framework-foundry-hosting >= 1.0.0a260630`), then `azd deploy`. Or pin the package to `1.0.0a260625` to stay on `1.0.0`. |
| `RuntimeError: ToolApprovalMiddleware requires an AgentSession` | A **harness agent** (`create_harness_agent`) enables tool approval by default, but the **Responses** hosting path runs without a per-request session. Pass `disable_tool_auto_approval=True` to `create_harness_agent` (hosted agents have no interactive approver), then `azd deploy`. Or serve via the **Invocations** protocol, which supplies sessions. |